# 06 - Per-Game Team Batting Logs
Fetch and cache per-game batting logs for all 30 MLB teams across 2015-2024.

**Data source:** Official MLB Stats API (`statsapi`) via the `team_stats` gameLog endpoint.
Replaces pybaseball `team_game_logs()`, which scraped Baseball Reference.
BRef now returns HTTP 403 (Cloudflare JS-challenge) for all programmatic requests.

300 calls total (30 teams x 10 seasons) at 0.3s courtesy delay.
Estimated time: ~2 minutes for uncached, instant for cached.


In [ ]:
import sys
sys.path.insert(0, "..")
from src.features.game_logs import fetch_all_team_game_logs, fetch_team_game_log, ALL_TEAMS
from src.data.cache import load_manifest
import logging
logging.basicConfig(level=logging.INFO)

In [ ]:
SEASONS = list(range(2015, 2026))
print(f"Fetching game logs for {len(ALL_TEAMS)} teams x {len(SEASONS)} seasons = {len(ALL_TEAMS) * len(SEASONS)} team-seasons")

In [ ]:
failures = fetch_all_team_game_logs(seasons=SEASONS, teams=ALL_TEAMS)
if failures:
    print(f"WARNING: {len(failures)} team-seasons failed to fetch: {failures}")
    print("These will appear as missing data in the feature matrix.")
else:
    print("Done! All team game logs cached successfully.")

In [ ]:
# Show sample data for one team-season
sample = fetch_team_game_log(2023, "NYY")
print(f"Sample: NYY 2023 - {len(sample)} games")
print(f"Columns: {list(sample.columns)}")
sample.head()

In [ ]:
# Verify team-seasons are cached, accounting for any failures surfaced above
manifest = load_manifest()
cached_count = sum(1 for k in manifest if k.startswith("team_game_log_"))
expected = len(SEASONS) * len(ALL_TEAMS)
print(f"Cached: {cached_count}/{expected} team-seasons")
if failures:
    print(f"Expected missing due to failures: {len(failures)}: {failures}")
    assert cached_count >= expected - len(failures), \
        f"More uncached than failures: {expected - cached_count} missing vs {len(failures)} failures"
else:
    assert cached_count == expected, f"Missing {expected - cached_count} team-seasons with no failures reported!"
    print("Coverage check PASSED: all 300 team-seasons cached.")

In [ ]:
# Spot-check a few entries
for season in [2015, 2020, 2024]:
    for team in ["NYY", "LAD", "CHC"]:
        df = fetch_team_game_log(season, team)
        expected_games = 60 if season == 2020 else 162
        # Allow some tolerance (not all teams play exactly 162)
        assert len(df) >= expected_games * 0.9, f"{team} {season}: only {len(df)} games"
        print(f"{team} {season}: {len(df)} games OK")